In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("../src")
# Geometry objectors
from hypo.lenspy import simple_Lens 
from hypo.surface import BiconicSurface,ConicSurface
from hypo.coordinate import coord_sys
# source objector
from hypo.Feedpy import GaussiBeam

# output objector
from hypo.field_storage import Spherical_grd, plane_grd, save_grd 
from hypo.coxvec import Ludwig_Cox_vector as CO
from hypo.vecops import Vector, dot

In [ ]:
'''1. Define coordinate system'''
coord_ref = coord_sys()


In [ ]:
coord_feed = coord_sys(ref_coord = coord_ref)
coord_Lens1 = coord_sys(origin = [0,0,100], ref_coord= coord_ref)
coord_Lens2 = coord_sys(origin = [0,0,100+6+260], ref_coord= coord_ref)
coord_out = coord_sys(origin = [0,0,100+6+260 +6+160], ref_coord =coord_ref)

In [ ]:
'''2. Define lens two surfaces'''
r1_x = 306.8 # mm
r1_y = 236
k1_x = -11.29
k1_y = -11.29
Lens1_face1 = BiconicSurface(r1_x, r1_y, conic_const_x=k1_x, conic_const_y=k1_y)
r1_2 = np.inf 
Lens1_face2 = ConicSurface(r1_2)


r2_2 = np.inf 
Lens2_face1 = ConicSurface(r2_2)
r2_x = 306.8 # mm
r2_y = 377.6
k2_x = -11.29
k2_y = -11.29
Lens2_face2 = BiconicSurface(r2_x, r2_y, conic_const_x=k2_x, conic_const_y=k2_y)

In [ ]:
'''3. Define Simple Lens'''
# refractive index
SILICON = 3.36
# thickness
t1 = 6 #mm
t2 = 6
# diameter
D  = 100# mm

Lens1 = simple_Lens(SILICON,
                    t1,
                    D,
                    Lens1_face1,
                    Lens1_face2,
                    coord_Lens1,
                    name = 'Lens1',
                    AR_file = None,
                    outputfolder = 'Data/biconic/')

Lens2 = simple_Lens(SILICON,
                    t2,
                    D,
                    Lens2_face1,
                    Lens2_face2,
                    coord_Lens2,
                    name = 'Lens2',
                    AR_file = None,
                    outputfolder = 'Data/biconic/')

In [ ]:
'''4. Source: an idea Gaussian beam'''
Edge_taper  = -3 #dB
Edge_angle = 17 # degree
freq = 150
Feed = GaussiBeam(Edge_taper, Edge_angle, freq, coord_feed,polarization='x')

In [ ]:
'''5. Define the fields wanted to calculated'''

OutputBeam = plane_grd(coord_out,
                        0,0,50,50,
                        101,101)

In [ ]:
''' Start PO anlaysis'''

Lens1.PO_analysis(Feed,
                  [1000,1000],
                  [1000,1000],
                  freq)
Lens2.PO_analysis(Lens1.source, 
                  [1000,1000],
                  [1000,1000],
                  freq)
Lens2.source(OutputBeam, freq, far_near = 'near')
save_grd(OutputBeam, 'Data/biconic/centerbeam.h5')

In [ ]:
r, theta, phi = Beammap.coord_sys.ToSpherical(Beammap.grid.x,Beammap.grid.y, Beammap.grid.z)
co,cx,crho = CO(theta,phi)
print(co)
E_co = dot(Beammap.E , co)
E_cx = dot(Beammap.E , cx)

In [ ]:
plt.pcolor(np.log10(np.abs(Beammap.E.y.reshape(1001,-1)))*10,cmap = 'jet',vmax = 30,vmin = -30)

In [ ]:
plt.pcolor(np.log10(np.abs(Beammap.E.x.reshape(1001,-1)))*10,cmap = 'jet',vmax = 30,vmin = -30)
plt.colorbar()

In [ ]:

plt.plot(np.log10(np.abs(E_co.reshape(1001,-1)[500,:]))*20)
plt.plot(np.log10(np.abs((E_cx.reshape(1001,-1))[500,:]))*20)
#plt.ylim([-80,30])
plt.grid()